# OQI Use Cases: Quantum Simulation for UN SDGs

Two use cases from the [Open Quantum Institute (OQI)](https://open-quantum-institute.cern/)
white paper, both addressable today with the uniqx chemistry SDK:

1. **SDG 12 — Coated fertiliser design** — quantum simulation of urea electronic
   structure, the foundation for understanding how coated fertilisers release nitrogen
   in soil while reducing run-off and N₂O emissions.
2. **SDG 3 — Drug metabolism design** — quantum simulation of formaldehyde, a key
   intermediate produced by CYP450 enzymes during oxidative N-demethylation. Predicting
   metabolite electronic structure feeds into early-stage drug-safety screening.

Both examples follow the same uniqx pattern: define a geometry, extract a basis,
trace `rhf_module`, submit, and inspect the result. All electronic-structure
computation runs server-side.

## Setup

In [ ]:
import os

from uniqx import connect, submit, get, parse_buffer_view, login
from uniqx.domains.chemistry.basis import extract_basis
from uniqx.domains.chemistry.hartree_fock import rhf_module

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)


def runtime_inputs_from(info):
    return [
        list(info.exps_flat),
        list(info.coeffs_flat),
        list(info.centers_flat),
        list(info.ang_flat),
        list(info.atom_coords_flat),
        list(info.charges_flat),
    ]


def run_scf(atoms, basis="sto-3g"):
    """Trace and submit an RHF calculation. Returns (info, energy_or_None).

    Returns ``(info, None)`` if the backend cannot return a parseable scalar
    energy for this molecule (a known limitation for some species/basis-set
    combinations on the current production endpoint).
    """
    info = extract_basis(atoms, basis)
    try:
        mod = rhf_module(atoms, info)
        job_id = submit(mod, client=client, runtime_inputs=runtime_inputs_from(info))
        result = get(job_id, client=client, timeout=300.0)
        payload = result.get("payload") or result.get("result_payload") or b""
        if isinstance(payload, str):
            payload = payload.encode("utf-8", errors="replace")
        for ln in payload.decode("utf-8", errors="replace").splitlines():
            try:
                parsed = parse_buffer_view(ln)
            except (ValueError, TypeError):
                continue
            if parsed is not None and parsed[2]:
                return info, parsed[2][-1]
        return info, None
    except (ValueError, RuntimeError):
        return info, None


## Use Case 1 — SDG 12: Coated Fertiliser Design (Urea)

**Goal**: simulate the electronic structure of urea (CO(NH₂)₂), the most widely used
nitrogen fertiliser globally. Controlled-release coated fertilisers encapsulate urea
in polymer shells to reduce nutrient run-off and N₂O emissions. Quantum simulation of
urea's electronic properties — bond polarisation, charge distribution, and (later)
NMR signatures — informs how urea interacts with coating polymers at the molecular
level.

**Molecule**: urea — 8 atoms (C, O, 2N, 4H). With STO-3G that is 24 basis functions
and 32 electrons.

In [ ]:
# Urea geometry (Å), planar C2v conformation.
UREA = [
    ("C", [0.0,    0.0,     0.000]),
    ("O", [0.0,    0.0,     1.221]),
    ("N", [1.166, 0.0,    -0.717]),
    ("N", [-1.166, 0.0,   -0.717]),
    ("H", [2.030, 0.0,    -0.196]),
    ("H", [1.207, 0.0,    -1.732]),
    ("H", [-2.030, 0.0,   -0.196]),
    ("H", [-1.207, 0.0,   -1.732]),
]

info, e_urea = run_scf(UREA, basis="sto-3g")
print(f"basis:       sto-3g")
print(f"n_atoms:     {info.n_atoms}")
print(f"n_basis:     {info.n_basis}")
print(f"n_electrons: {info.n_electrons}")
if e_urea is None:
    print("E(urea):     SCF on urea hit a backend issue (Known limitation)")
else:
    print(f"E(urea):     {e_urea:.6f} Ha")


## Use Case 2 — SDG 3: Drug Metabolism Design (Formaldehyde)

**Goal**: simulate formaldehyde (H₂CO), a key metabolic intermediate produced by
cytochrome P450 (CYP450) enzymes during oxidative N-demethylation of drugs.
Understanding the electronic structure of metabolic intermediates helps predict which
drug candidates will produce toxic metabolites — a contributor to adverse drug
reactions, one of the leading causes of preventable hospitalisations.

**Molecule**: formaldehyde — 4 atoms (C, O, 2H). With STO-3G that is 12 basis
functions and 16 electrons.

In [ ]:
# Formaldehyde geometry (Å).
FORMALDEHYDE = [
    ("C", [0.0,  0.0,    0.000]),
    ("O", [0.0,  0.0,    1.220]),
    ("H", [0.0,  0.945, -0.587]),
    ("H", [0.0, -0.945, -0.587]),
]

info, e_form = run_scf(FORMALDEHYDE, basis="sto-3g")
print(f"basis:       sto-3g")
print(f"n_atoms:     {info.n_atoms}")
print(f"n_basis:     {info.n_basis}")
print(f"n_electrons: {info.n_electrons}")
if e_form is None:
    print("E(formaldehyde): SCF on formaldehyde hit a backend issue (Known limitation)")
else:
    print(f"E(formaldehyde): {e_form:.6f} Ha")


## Discussion

Both molecules are at the small end of what STO-3G + RHF can describe — useful for
illustrating the API and for benchmarking against larger basis sets. For research-grade
predictions:

- Switch the basis to `cc-pvdz` (or larger) by changing the `basis=` argument to
  `extract_basis`.
- Use `nmr_full_module` instead of `rhf_module` to add isotropic shieldings and
  Fermi-contact J-couplings (see the `nmr_notebook` example).
- For larger systems (~30+ atoms), call `preflight(...)` first to see how uniqx will
  schedule the work across CPU / GPU / QPU options.

The same workflow scales to dozens of atoms; the only thing that changes is the
geometry list and the basis-set string.